# Olist Data Analysis with PySpark

This notebook uses the same loading pattern as Cell 34 (multiple `spark.read.csv(..., header=True)` calls), but applies it to the datasets in `notebook/oist-data`.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('olist-analysis').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/14 14:59:17 WARN Utils: Your hostname, frethinkstation, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/14 14:59:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/14 14:59:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/14 14:59:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# Optional: keep behavior consistent with the class notebook
spark.conf.set('spark.sql.ansi.enabled', 'false')

In [ ]:
# Download and extract all required Olist CSV files into this folder before proceeding. 
# Refer to Readme for details.

data_dir = 'oist-data'

In [4]:
# Same style as Cell 34: load multiple CSV files with Spark
customers = spark.read.csv(f'{data_dir}/olist_customers_dataset.csv', header=True)
geolocation = spark.read.csv(f'{data_dir}/olist_geolocation_dataset.csv', header=True)
order_items = spark.read.csv(f'{data_dir}/olist_order_items_dataset.csv', header=True)
order_payments = spark.read.csv(f'{data_dir}/olist_order_payments_dataset.csv', header=True)
order_reviews = spark.read.csv(f'{data_dir}/olist_order_reviews_dataset.csv', header=True)
orders = spark.read.csv(f'{data_dir}/olist_orders_dataset.csv', header=True)
products = spark.read.csv(f'{data_dir}/olist_products_dataset.csv', header=True)
sellers = spark.read.csv(f'{data_dir}/olist_sellers_dataset.csv', header=True)
category_translation = spark.read.csv(f'{data_dir}/product_category_name_translation.csv', header=True)

In [ ]:
datasets = {
    'customers': customers,
    'geolocation': geolocation,
    'order_items': order_items,
    'order_payments': order_payments,
    'order_reviews': order_reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'category_translation': category_translation
}

list(datasets.keys())

In [ ]:
# Quick shape check for each dataset
for name, df in datasets.items():
    print(f'\n{name}:')
    print(f'rows = {df.count()}, cols = {len(df.columns)}')

In [ ]:
# Inspect schemas
orders.printSchema()
order_items.printSchema()
products.printSchema()

In [ ]:
# Preview key tables
orders.show(5, truncate=False)
order_items.show(5, truncate=False)
products.show(5, truncate=False)

In [ ]:
# Example analysis 1: order status distribution
orders.groupBy('order_status').count().orderBy('count', ascending=False).show()

In [ ]:
# Example analysis 2: payment type distribution
order_payments.groupBy('payment_type').count().orderBy('count', ascending=False).show()

In [ ]:
# Example analysis 3: top categories by item count
top_categories = (
    order_items
    .join(products, on='product_id', how='left')
    .join(category_translation, on='product_category_name', how='left')
    .groupBy('product_category_name', 'product_category_name_english')
    .count()
    .orderBy('count', ascending=False)
)

top_categories.show(100, truncate=False)

## KPI Analysis

This section computes:
1. Monthly sales trends
2. Top-selling products
3. Customer segmentation by purchase behavior

In [ ]:
from pyspark.sql import functions as F

# Build typed datasets for reliable aggregations and time analysis.
orders_typed = (
    orders
    .withColumn('order_purchase_ts', F.to_timestamp('order_purchase_timestamp'))
    .withColumn('order_delivered_customer_ts', F.to_timestamp('order_delivered_customer_date'))
)

order_items_typed = (
    order_items
    .withColumn('price_num', F.col('price').cast('double'))
    .withColumn('freight_num', F.col('freight_value').cast('double'))
)

order_value_by_order = (
    order_items_typed
    .groupBy('order_id')
    .agg(
        F.sum(F.coalesce(F.col('price_num'), F.lit(0.0)) + F.coalesce(F.col('freight_num'), F.lit(0.0))).alias('order_value')
    )
)

In [ ]:
# 1) Monthly sales trends (delivered orders only).
monthly_sales = (
    orders_typed
    .filter(F.col('order_status') == 'delivered')
    .join(order_value_by_order, on='order_id', how='left')
    .withColumn('order_month', F.date_format('order_purchase_ts', 'yyyy-MM'))
    .groupBy('order_month')
    .agg(
        F.countDistinct('order_id').alias('total_orders'),
        F.round(F.sum(F.coalesce(F.col('order_value'), F.lit(0.0))), 2).alias('total_sales')
    )
    .orderBy('order_month')
)

monthly_sales.show(100, truncate=False)

In [ ]:
# 2) Top-selling product categories by quantity and revenue.
top_selling_products = (
    order_items_typed
    .join(products, on='product_id', how='left')
    .join(category_translation, on='product_category_name', how='left')
    .groupBy('product_category_name_english')
    .agg(
        F.count('*').alias('units_sold'),
        F.round(F.sum(F.coalesce(F.col('price_num'), F.lit(0.0))), 2).alias('product_revenue')
    )
    .orderBy(F.desc('product_revenue'), F.desc('units_sold'))
)

top_selling_products.show(25, truncate=False)

In [ ]:
# 3) Customer segmentation by purchase behavior.
customer_metrics = (
    orders_typed
    .filter(F.col('order_status') == 'delivered')
    .join(order_value_by_order, on='order_id', how='left')
    .groupBy('customer_id')
    .agg(
        F.countDistinct('order_id').alias('frequency'),
        F.round(F.sum(F.coalesce(F.col('order_value'), F.lit(0.0))), 2).alias('monetary'),
        F.max('order_purchase_ts').alias('last_purchase_ts')
    )
    .withColumn('recency_days', F.datediff(F.current_date(), F.to_date('last_purchase_ts')))
    .filter(F.col('last_purchase_ts').isNotNull())
)

# Use distribution cutoffs to form behavior-based segments.
m_cutoffs = customer_metrics.approxQuantile('monetary', [0.33, 0.66], 0.01)
r_cutoffs = customer_metrics.approxQuantile('recency_days', [0.33, 0.66], 0.01)
m_low, m_high = m_cutoffs[0], m_cutoffs[1]
r_low, r_high = r_cutoffs[0], r_cutoffs[1]

customer_segments = (
    customer_metrics
    .withColumn(
        'frequency_segment',
        F.when(F.col('frequency') >= 5, 'Loyal')
         .when(F.col('frequency') >= 2, 'Repeat')
         .otherwise('One-time')
    )
    .withColumn(
        'spend_segment',
        F.when(F.col('monetary') >= F.lit(m_high), 'High Spend')
         .when(F.col('monetary') >= F.lit(m_low), 'Mid Spend')
         .otherwise('Low Spend')
    )
    .withColumn(
        'recency_segment',
        F.when(F.col('recency_days') <= F.lit(r_low), 'Recent')
         .when(F.col('recency_days') <= F.lit(r_high), 'Warm')
         .otherwise('Dormant')
    )
)

segment_summary = (
    customer_segments
    .groupBy('frequency_segment', 'spend_segment', 'recency_segment')
    .agg(F.count('*').alias('customers'))
    .orderBy(F.desc('customers'))
)

segment_summary.show(30, truncate=False)

## KPI Visualizations

Line and bar charts for the KPI outputs.

In [ ]:
import matplotlib.pyplot as plt

monthly_sales_pd = monthly_sales.toPandas()
monthly_sales_pd['total_sales'] = monthly_sales_pd['total_sales'].astype(float)

plt.figure(figsize=(12, 5))
plt.plot(monthly_sales_pd['order_month'], monthly_sales_pd['total_sales'], marker='o')
plt.xticks(rotation=45, ha='right')
plt.title('Monthly Sales Trend (Delivered Orders)')
plt.xlabel('Month')
plt.ylabel('Total Sales')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
top_categories_pd = (
    top_selling_products
    .filter(F.col('product_category_name_english').isNotNull())
    .groupBy('product_category_name_english')
    .agg(
        F.sum('units_sold').alias('units_sold'),
        F.round(F.sum('product_revenue'), 2).alias('category_revenue')
    )
    .orderBy('units_sold', ascending=False)
    .limit(10)
    .toPandas()
)

top_categories_pd['category_label'] = (
    top_categories_pd['product_category_name_english']
    .str.replace('_', ' ', regex=False)
    .str.title()
)
top_categories_pd = top_categories_pd.sort_values('units_sold', ascending=True)

plt.figure(figsize=(12, 6))
plt.barh(top_categories_pd['category_label'], top_categories_pd['units_sold'])
plt.title('Top 10 Product Categories by Units Sold')
plt.xlabel('Units Sold')
plt.ylabel('Product Category')
plt.tight_layout()
plt.show()

In [ ]:
top_revenue_categories_pd = (
    top_selling_products
    .filter(F.col('product_category_name_english').isNotNull())
    .groupBy('product_category_name_english')
    .agg(F.round(F.sum('product_revenue'), 2).alias('category_revenue'))
    .orderBy('category_revenue', ascending=False)
    .limit(15)
    .toPandas()
)

top_revenue_categories_pd['category_revenue'] = top_revenue_categories_pd['category_revenue'].astype(float)
top_revenue_categories_pd['category_label'] = (
    top_revenue_categories_pd['product_category_name_english']
    .str.replace('_', ' ', regex=False)
    .str.title()
)
top_revenue_categories_pd = top_revenue_categories_pd.sort_values('category_revenue', ascending=True)

plt.figure(figsize=(12, 7))
bars = plt.barh(
    top_revenue_categories_pd['category_label'],
    top_revenue_categories_pd['category_revenue']
 )

for bar, value in zip(bars, top_revenue_categories_pd['category_revenue']):
    plt.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f' ${value:,.2f}',
        va='center',
        ha='left'
    )

plt.title('Top 15 Product Categories by Revenue')
plt.xlabel('Revenue (in millions) $')
plt.ylabel('Product Category')
plt.tight_layout()
plt.show()

In [ ]:
null_category_breakdown = (
    order_items
    .join(products, on='product_id', how='left')
    .join(category_translation, on='product_category_name', how='left')
    .withColumn(
        'null_category_reason',
        F.when(F.col('product_category_name').isNull(), 'missing source category')
         .when(F.col('product_category_name_english').isNull(), 'missing translation')
         .otherwise('mapped')
    )
    .groupBy('null_category_reason')
    .count()
    .orderBy('count', ascending=False)
)

null_category_breakdown.show(truncate=False)